## Variable Selection and Feature Construction

**Raw data source:** Swiss NABEL air quality monitoring network
[https://www.bafu.admin.ch/en/data-query-nabel](https://www.bafu.admin.ch/en/data-query-nabel)

### i. Original Predictors in the Dataset

The merged daily dataset contains the following measured variables:

* **PM10 (µg/m³)** – particulate matter concentration
* **NO₂ (µg/m³)** – traffic-related pollutant
* **O₃ (µg/m³)** – secondary photochemical pollutant
* **Temperature (°C)** – atmospheric conditions
* **Precipitation (mm)** – wet deposition mechanism
* **Global radiation (W/m²)** – solar energy input

These variables were selected because they jointly represent emission intensity (NO₂), atmospheric chemistry (O₃, radiation), and meteorological dispersion or removal processes (temperature, precipitation). Together they explain short-term variation in particulate matter concentrations.

### ii. Response Variable

The main response variable is **PM10 (µg/m³)**, a continuous air pollution indicator with direct health relevance.

Since PM10 is non-negative and right-skewed, it is transformed as:

[
\texttt{log_pm10} = \log(1 + \texttt{pm10})
]

This transformation stabilizes variance and satisfies modelling assumptions for linear models (LM), generalized additive models (GAM), neural networks, and SVM regression.

### iii. Derived Variables

To satisfy the required response types and modelling structures, additional variables are constructed from the original PM10 and date information.

#### (a) Binary Response

* **`pm10_high`**

  * Defined as:
    1 if `pm10 > 50 µg/m³` (Swiss daily limit),
    0 otherwise.
  * Created directly from the original **PM10** variable.
  * Used for Binomial GLM (logistic regression).

#### (b) Count Variable

* **`n_exceed` / `exceed_mtd`**

  * Monthly count (or cumulative month-to-date count) of days with `pm10 > 50 µg/m³`.
  * Derived from `pm10_high` aggregated by station and month.
  * Used for Poisson GLM.

### iv. Time-Based Variables

To capture temporal structure, additional predictors are generated from the original **date** variable:

* **`year`** – long-term trend
* **`month`** (categorical) – seasonality
* **`wday`** (categorical) – weekly pattern
* **`weekend`** (binary; derived from weekday) – activity differences

Optionally, smooth seasonality is represented by:

* **`sin_doy` and `cos_doy`**, calculated from the day-of-year to encode cyclic annual patterns.

### v. Spatial Variables

To account for systematic differences between monitoring locations:

* **`station`** (categorical; directly available in dataset)
* **`station_type`** (derived from station classification: urban, suburban, rural, traffic)
* **`altitude`** (mapped from station metadata)

These variables capture persistent structural differences in emission environment and atmospheric conditions across sites.

# 1. Imports and Environment Setup


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/air_quality"


pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1400)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2. Loading the Merged Panel Dataset



In [2]:
FILE_PATH = os.path.join(DATA_DIR, "panel_merged_2005_2025.csv")

df = pd.read_csv(FILE_PATH)

# Convert date column
df["date"] = pd.to_datetime(df["date"])

print("Shape:", df.shape)
display(df.head())

Shape: (99710, 8)


,date,station,NO2 [ug/m3],O3 [ug/m3],PM10 [ug/m3],PREC [mm],RAD [W/m2],TEMP [C]
0,2005-01-01,Basel-Binningen,40.0,14.2,44.7,0.9,30.0,3.7
1,2005-01-02,Basel-Binningen,11.7,76.2,7.8,6.4,14.0,5.7
2,2005-01-03,Basel-Binningen,14.6,64.3,12.2,0.0,42.7,3.7
3,2005-01-04,Basel-Binningen,44.2,29.9,20.0,0.0,66.4,1.9
4,2005-01-05,Basel-Binningen,43.6,66.1,24.3,0.7,27.4,3.5


# 3. Post-Merge Validation

In [3]:
# --- Ensure types ---
df = df.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# --- Define columns ---
KEY_COLS = ["date", "station"]
VALUE_COLS = [c for c in df.columns if c not in KEY_COLS]

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (99710, 8)
Columns: ['date', 'station', 'NO2 [ug/m3]', 'O3 [ug/m3]', 'PM10 [ug/m3]', 'PREC [mm]', 'RAD [W/m2]', 'TEMP [C]']


# 4. Key Integrity Checks

4.1 Missing Keys

In [4]:
print("\n--- Missing keys ---")
print("Missing date:", df["date"].isna().sum())
print("Missing station:", df["station"].isna().sum())


--- Missing keys ---
Missing date: 0
Missing station: 0


4.2 Duplicate Observations

In [5]:
dup_n = df.duplicated(subset=KEY_COLS).sum()
print("\n--- Duplicate (date, station) rows ---")
print("Duplicates:", dup_n)


--- Duplicate (date, station) rows ---
Duplicates: 0


4.3 Date Range

In [6]:
print("\n--- Date range ---")
print(df["date"].min(), "->", df["date"].max())


--- Date range ---
2005-01-01 00:00:00 -> 2025-12-31 00:00:00


4.4 Station Count

In [7]:
print("\n--- Stations ---")
print("n_stations:", df["station"].nunique())
print("Stations:", sorted(df["station"].unique())[:10], "...")


--- Stations ---
n_stations: 13
Stations: ['Basel-Binningen', 'Bern-Bollwerk', 'Chaumont', 'Dübendorf-Empa', 'Härkingen-A1', 'Lausanne-César-Roux', 'Lugano-Università', 'Magadino-Cadenazzo', 'Payerne', 'Rigi-Seebodenalp'] ...


# 5. Missingness Diagnostics

Analyses performed:

*   Missing counts per variable
*   Percentage missing per variable
*   Missingness by station

In [8]:
print("\n--- Missingness by variable ---")
missing_n = df[VALUE_COLS].isna().sum().sort_values(ascending=False)
missing_pct = (df[VALUE_COLS].isna().mean() * 100).sort_values(ascending=False)
miss_summary = pd.DataFrame({"missing_n": missing_n, "missing_%": missing_pct.round(2)})
display(miss_summary)


--- Missingness by variable ---


,missing_n,missing_%
O3 [ug/m3],936,0.94
PM10 [ug/m3],750,0.75
PREC [mm],561,0.56
NO2 [ug/m3],399,0.40
RAD [W/m2],397,0.40
TEMP [C],184,0.18


In [9]:
print("\n--- Missingness by station (avg % over variables) ---")
station_missing_pct = (
    df.groupby("station")[VALUE_COLS]
      .apply(lambda x: x.isna().mean().mean() * 100)
      .sort_values(ascending=False)
)
display(station_missing_pct)


--- Missingness by station (avg % over variables) ---


,0
station,
Rigi-Seebodenalp,0.916993
Chaumont,0.736636
Bern-Bollwerk,0.734463
Zürich-Kaserne,0.730117
Härkingen-A1,0.662755
Lausanne-César-Roux,0.575837
Lugano-Università,0.486745
Dübendorf-Empa,0.473707
Payerne,0.445458


# 6. Value Sanity Checks

In [10]:
print("\n--- Basic numeric summary ---")
display(df[VALUE_COLS].describe())


--- Basic numeric summary ---


,NO2 [ug/m3],O3 [ug/m3],PM10 [ug/m3],PREC [mm],RAD [W/m2],TEMP [C]
count,99311.000000,98774.000000,98960.000000,99149.000000,99313.000000,99526.000000
mean,22.482279,81.039914,16.036015,2.910248,145.691966,10.785414
std,16.539251,34.492629,12.163712,7.460428,100.069916,7.708158
min,0.000000,1.100000,0.000000,0.000000,-1.000000,-16.900000
25%,8.800000,59.000000,8.000000,0.000000,58.200000,4.800000
50%,18.600000,81.000000,13.100000,0.000000,125.000000,10.800000
75%,32.800000,102.700000,20.500000,2.200000,227.900000,16.900000
max,104.800000,199.500000,195.100000,175.300000,403.900000,30.300000


# 7. Data Quality Adjustments

The descriptive statistics reveal minor data issues that require correction before modelling:

* **PM10 contains zero values.** Since the logarithm of zero is undefined, we apply a log-transformation using `log1p(pm10)` (i.e., log(1 + PM10)). This preserves all observations while avoiding infinite values and stabilizing variance for linear models.

* **Radiation (RAD) includes a negative value (−1 W/m²).** Solar radiation cannot be negative, so these values are treated as missing (`NaN`). This prevents physically implausible values from biasing model estimates.

* **Precipitation is highly right-skewed with many zeros.** To reduce skewness and improve linear model behavior, we use `log1p(prec)`.

All corrections maintain interpretability while ensuring valid statistical assumptions for subsequent modelling steps.


In [11]:
# 1. Fix negative radiation values
df.loc[df["RAD [W/m2]"] < 0, "RAD [W/m2]"] = np.nan

# 2. Safe log transformation for PM10
df["log_pm10"] = np.log1p(df["PM10 [ug/m3]"])

# 3. Precipitation transformation
df["log_prec"] = np.log1p(df["PREC [mm]"])

# Check
print("Remaining negative radiation values:",
      (df["RAD [W/m2]"] < 0).sum())

Remaining negative radiation values: 0


# 8. Handling Missing Values

The dataset contains a very small proportion of missing values (all variables < 1%). Since most statistical and machine learning models cannot handle missing observations directly, and to ensure a fair comparison across models, we use a complete-case approach.

This means that we remove any row (i.e., station–day observation) where at least one of the required variables is missing. Given the low missingness, this leads to minimal data loss while keeping the modelling process simple, consistent, and reproducible.

In [12]:
# Work on a copy
df_base = df.copy()

# Ensure correct date format
df_base["date"] = pd.to_datetime(df_base["date"])

# Rename columns for clarity (no transformations yet)
df_base = df_base.rename(columns={
    "PM10 [ug/m3]": "pm10",
    "NO2 [ug/m3]": "no2",
    "O3 [ug/m3]": "o3",
    "PREC [mm]": "prec",
    "RAD [W/m2]": "rad",
    "TEMP [C]": "temp"
})

# Required base variables
required_cols = ["date", "station", "pm10", "no2", "o3", "temp", "prec", "rad"]

# Drop rows with missing values
df_base = df_base.dropna(subset=required_cols).reset_index(drop=True)

print("Validated base dataset shape:", df_base.shape)
display(df_base.head())
display(df_base.info())

Validated base dataset shape: (97282, 10)


,date,station,no2,o3,pm10,prec,rad,temp,log_pm10,log_prec
0,2005-01-01,Basel-Binningen,40.0,14.2,44.7,0.9,30.0,3.7,3.822098,0.641854
1,2005-01-02,Basel-Binningen,11.7,76.2,7.8,6.4,14.0,5.7,2.174752,2.001480
2,2005-01-03,Basel-Binningen,14.6,64.3,12.2,0.0,42.7,3.7,2.580217,0.000000
3,2005-01-04,Basel-Binningen,44.2,29.9,20.0,0.0,66.4,1.9,3.044522,0.000000
4,2005-01-05,Basel-Binningen,43.6,66.1,24.3,0.7,27.4,3.5,3.230804,0.530628


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97282 entries, 0 to 97281
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   date      97282 non-null  datetime64[ns]
 1   station   97282 non-null  object        
 2   no2       97282 non-null  float64       
 3   o3        97282 non-null  float64       
 4   pm10      97282 non-null  float64       
 5   prec      97282 non-null  float64       
 6   rad       97282 non-null  float64       
 7   temp      97282 non-null  float64       
 8   log_pm10  97282 non-null  float64       
 9   log_prec  97282 non-null  float64       
dtypes: datetime64[ns](1), float64(8), object(1)
memory usage: 7.4+ MB


None

# 9. Feature Engineering

To match the modelling requirements (continuous, binary, count, and categorical predictors with **>2 levels**), we extend the validated daily dataset with (i) response variables used by the different model families, (ii) time-based predictors capturing trends/seasonality, and (iii) **location/station variables** capturing systematic differences between measurement sites.

#### Response variables

* **`log_pm10`** – Log-transformed PM10 concentration for **Linear Models** and **GAMs**:
  `log_pm10 = log(pm10)`
  (PM10 is a positive “amount”, so log-stabilisation is typically appropriate.)

* **`pm10_high`** – Binary exceedance indicator for **Binomial GLM**:
  Swiss daily mean limit: **50 µg/m³**
  `pm10_high = 1` if `pm10 > 50`, else `0`.

* **`n_exceed`** – Monthly exceedance count per station for **Poisson GLM**:
  number of days in a month (per station) with `pm10 > 50`.

#### Time-based predictors

* **`year`** – Long-term trend.
* **`month`** (factor) – Seasonal variation (12 levels).
* **`wday`** (factor) – Weekly pattern (7 levels).
* **`weekend`** (binary) – Traffic/activity differences.
* **`sin_doy` / `cos_doy`** – Cyclical encoding of day-of-year for smooth seasonality (useful for NN/SVM and as alternatives to factor seasonality).

#### Location / station predictors (to satisfy “categorical with >2 levels” and capture site heterogeneity)
https://www.empa.ch/documents/56101/246436/F-NABEL-Messkonzept_2015.pdf/3a760049-f5c5-4708-9437-69d54e6469f2
https://www.bafu.admin.ch/dam/en/sd-web/ZzGRUJw5JAjU/nabel-messstationen.pdf
* **`station`** (factor) – Strong “fixed effect” capturing persistent station differences (many levels; often very predictive).
* **`station_type`** (factor, >2 levels) – e.g., traffic / urban background / rural / industrial (if available).
* **`altitude_m`** (numeric) – Continuous site characteristic (if available).



In [13]:
df_fe = df_base.copy()

df_fe = df_fe.sort_values(["station", "date"])

## 9.1 Continuous Response Variable

`log_pm10`

Used for:

* LM
* GAM
* Neural networks
* SVM regression

Log transformation stabilizes variance.


In [14]:
df_fe["log_pm10"] = np.log1p(df_fe["pm10"])   # safe for zeros

## 9.2 Binary Response Variable

Swiss daily PM10 limit: **50 µg/m³**

We define:

`pm10_high = 1` if pm10 > 50
`pm10_high = 0` otherwise

Used for:

* Logistic regression (Binomial GLM)

In [15]:
PM10_LIMIT = 50.0
df_fe["pm10_high"] = (df_fe["pm10"] > PM10_LIMIT).astype(int)

## 9.3 Count Variable

We compute month-to-date exceedance counts:

`exceed_mtd`

Definition:
Cumulative number of exceedance days within each station-month
(excluding the current day)

Used for:

* Poisson GLM
* Count-based modelling

In [16]:
# Create calendar month identifier
df_fe["month_start"] = df_fe["date"].dt.to_period("M").dt.to_timestamp()

# Compute cumulative exceedances within each station-month
df_fe["exceed_mtd"] = (
    df_fe.groupby(["station", "month_start"])["pm10_high"]
        .cumsum()
)

# Exclude today's exceedance from the count
df_fe["exceed_mtd"] = (
    df_fe.groupby(["station", "month_start"])["exceed_mtd"]
        .shift(1)
        .fillna(0)
        .astype(int)
)

# 10. Time-Based Predictors

To capture temporal structure:

### Long-Term Trend

* `year`

### Seasonal Effects

* `month` (12-level categorical)

### Weekly Pattern

* `wday`
* `weekend`

### Cyclical Encoding

* `sin_doy`
* `cos_doy`

Cyclical encoding ensures smooth transitions between December and January.


In [17]:
df_fe["year"] = df_fe["date"].dt.year
df_fe["month"] = df_fe["date"].dt.month.astype("category")

# Monday=0 ... Sunday=6  (convert to labels for readability)
wday_num = df_fe["date"].dt.dayofweek
df_fe["wday"] = wday_num.map({0:"Mon",1:"Tue",2:"Wed",3:"Thu",4:"Fri",5:"Sat",6:"Sun"}).astype("category")

df_fe["weekend"] = (wday_num >= 5).astype(int)

# Cyclical day-of-year encoding
doy = df_fe["date"].dt.dayofyear
df_fe["sin_doy"] = np.sin(2 * np.pi * doy / 365.25)
df_fe["cos_doy"] = np.cos(2 * np.pi * doy / 365.25)

# 11. Spatial and Station-Level Variables

To incorporate environmental heterogeneity:

### Station (Categorical)

Captures fixed site-specific differences.

### Station Type (Categorical > 2 Levels)

Categories include:

* Urban traffic
* Urban background
* Suburban
* Rural motorway
* Rural high altitude

This satisfies the multi-level categorical requirement.

### Altitude (Continuous)

Captures atmospheric and dispersion differences.

In [18]:
station_type_map = {
    # Urban, traffic
    "Bern-Bollwerk": "urban_traffic",
    "Lausanne-César-Roux": "urban_traffic",

    # Urban
    "Zürich-Kaserne": "urban",
    "Lugano-Università": "urban",

    # Suburban
    "Basel-Binningen": "suburban",
    "Dübendorf-Empa": "suburban",

    # Rural, motorway
    "Härkingen-A1": "rural_motorway",
    "Sion-Aéroport-A9": "rural_motorway",

    # Rural, altitude < 1000 m
    "Magadino-Cadenazzo": "rural_low_altitude",
    "Payerne": "rural_low_altitude",
    "Tänikon": "rural_low_altitude",

    # Rural, altitude > 1000 m
    "Chaumont": "rural_high_altitude",
    "Rigi-Seebodenalp": "rural_high_altitude"
}

altitude_map = {
    "Rigi-Seebodenalp": 1031,
    "Chaumont": 1136,
    "Bern-Bollwerk": 536,
    "Zürich-Kaserne": 409,
    "Härkingen-A1": 431,
    "Lausanne-César-Roux": 530,
    "Lugano-Università": 280,
    "Dübendorf-Empa": 432,
    "Payerne": 489,
    "Magadino-Cadenazzo": 203,
    "Tänikon": 538,
    "Basel-Binningen": 316,
    "Sion-Aéroport-A9": 483
}

# Apply mappings to df_fe

df_fe["station_type"] = df_fe["station"].map(station_type_map)
df_fe["station_type"] = df_fe["station_type"].astype("category")

df_fe["altitude"] = df_fe["station"].map(altitude_map)

# Validation check

print("Missing station_type:", df_fe["station_type"].isna().sum())
print("Missing altitude:", df_fe["altitude"].isna().sum())

display(df_fe.head())

Missing station_type: 0
Missing altitude: 0


,date,station,no2,o3,pm10,prec,rad,temp,log_pm10,log_prec,pm10_high,month_start,exceed_mtd,year,month,wday,weekend,sin_doy,cos_doy,station_type,altitude
0,2005-01-01,Basel-Binningen,40.0,14.2,44.7,0.9,30.0,3.7,3.822098,0.641854,0,2005-01-01,0,2005,1,Sat,1,0.017202,0.999852,suburban,316
1,2005-01-02,Basel-Binningen,11.7,76.2,7.8,6.4,14.0,5.7,2.174752,2.001480,0,2005-01-01,0,2005,1,Sun,1,0.034398,0.999408,suburban,316
2,2005-01-03,Basel-Binningen,14.6,64.3,12.2,0.0,42.7,3.7,2.580217,0.000000,0,2005-01-01,0,2005,1,Mon,0,0.051584,0.998669,suburban,316
3,2005-01-04,Basel-Binningen,44.2,29.9,20.0,0.0,66.4,1.9,3.044522,0.000000,0,2005-01-01,0,2005,1,Tue,0,0.068755,0.997634,suburban,316
4,2005-01-05,Basel-Binningen,43.6,66.1,24.3,0.7,27.4,3.5,3.230804,0.530628,0,2005-01-01,0,2005,1,Wed,0,0.085906,0.996303,suburban,316


In [19]:
print("Daily feature dataset shape:", df_fe.shape)
display(df_fe.head())
display(df_fe.info())

Daily feature dataset shape: (97282, 21)


,date,station,no2,o3,pm10,prec,rad,temp,log_pm10,log_prec,pm10_high,month_start,exceed_mtd,year,month,wday,weekend,sin_doy,cos_doy,station_type,altitude
0,2005-01-01,Basel-Binningen,40.0,14.2,44.7,0.9,30.0,3.7,3.822098,0.641854,0,2005-01-01,0,2005,1,Sat,1,0.017202,0.999852,suburban,316
1,2005-01-02,Basel-Binningen,11.7,76.2,7.8,6.4,14.0,5.7,2.174752,2.001480,0,2005-01-01,0,2005,1,Sun,1,0.034398,0.999408,suburban,316
2,2005-01-03,Basel-Binningen,14.6,64.3,12.2,0.0,42.7,3.7,2.580217,0.000000,0,2005-01-01,0,2005,1,Mon,0,0.051584,0.998669,suburban,316
3,2005-01-04,Basel-Binningen,44.2,29.9,20.0,0.0,66.4,1.9,3.044522,0.000000,0,2005-01-01,0,2005,1,Tue,0,0.068755,0.997634,suburban,316
4,2005-01-05,Basel-Binningen,43.6,66.1,24.3,0.7,27.4,3.5,3.230804,0.530628,0,2005-01-01,0,2005,1,Wed,0,0.085906,0.996303,suburban,316


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97282 entries, 0 to 97281
Data columns (total 21 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          97282 non-null  datetime64[ns]
 1   station       97282 non-null  object        
 2   no2           97282 non-null  float64       
 3   o3            97282 non-null  float64       
 4   pm10          97282 non-null  float64       
 5   prec          97282 non-null  float64       
 6   rad           97282 non-null  float64       
 7   temp          97282 non-null  float64       
 8   log_pm10      97282 non-null  float64       
 9   log_prec      97282 non-null  float64       
 10  pm10_high     97282 non-null  int64         
 11  month_start   97282 non-null  datetime64[ns]
 12  exceed_mtd    97282 non-null  int64         
 13  year          97282 non-null  int32         
 14  month         97282 non-null  category      
 15  wday          97282 non-null  catego

None

# 12. Final Modeling Dataset Construction

We select the final modelling variables.

Final dataset characteristics:

* 97,282 observations
* 15 variables
* Mixed variable types

Includes:

* Continuous predictors
* Multi-level categorical predictors
* Binary response
* Count variable


In [20]:
# --- choose columns to keep (recommended: includes continuous response) ---
cols_to_keep = [
    # identifiers / grouping
    "date",
    "station",
    "station_type",
    "altitude",

    # meteorology & pollutants (predictors)
    "no2",
    "o3",
    "rad",
    "temp",
    "log_prec",

    # time features (predictors)
    "year",
    "month",
    "wday",
    "weekend",
    "sin_doy",
    "cos_doy",

    # responses (keep all three; you can choose per model later)
    "log_pm10",     # continuous response for LM/GAM/SVM/NN reg
    "pm10_high",    # binary response for Binomial GLM
    "exceed_mtd"    # count response for Poisson GLM
]

df_final = df_fe.loc[:, cols_to_keep].copy()

# --- optional: enforce types for consistency ---
df_final["station"] = df_final["station"].astype("category")
df_final["weekend"] = df_final["weekend"].astype("int8")
df_final["pm10_high"] = df_final["pm10_high"].astype("int8")
df_final["exceed_mtd"] = df_final["exceed_mtd"].astype("int16")
df_final["year"] = df_final["year"].astype("int16")

print("df_final shape:", df_final.shape)
print(df_final.dtypes)

df_final shape: (97282, 18)
date            datetime64[ns]
station               category
station_type          category
altitude                 int64
no2                    float64
o3                     float64
rad                    float64
temp                   float64
log_prec               float64
year                     int16
month                 category
wday                  category
weekend                   int8
sin_doy                float64
cos_doy                float64
log_pm10               float64
pm10_high                 int8
exceed_mtd               int16
dtype: object


### Final Dataset Validation

The final modelling dataset `df_final` contains **97,282 observations** and **15 variables** (dimension **97,282 × 15**). It satisfies the professor’s stated requirements regarding dataset scale, real-world origin, and the inclusion of multiple response and predictor types.

#### 1. Dataset size

* **N = 97,282**, which falls within the required range **10³ ≤ N ≤ 10⁵**.
* **15 variables**, matching the recommended range of roughly **10–20 variables**.

#### 2. Real-world data source

The dataset is constructed from **official Swiss NABEL air quality monitoring data**, consisting of daily measurements at multiple monitoring stations over a long time period. This ensures the data is real-world and environmentally meaningful.

#### 3. Required variable types (all included in one dataset)

The dataset contains all required variable types simultaneously:

* **Continuous variables:**
  `no2`, `o3`, `rad`, `temp`, `log_prec`, `sin_doy`, `cos_doy`

* **Categorical variables (with more than two levels):**
  `station`, `wday`, `station_type`

* **Binary variable:**
  `pm10_high` (daily exceedance indicator)

* **Count variable:**
  `exceed_mtd` (month-to-date number of exceedance days per station)

#### 4. Suitability for GLM and related modelling

The dataset supports the planned modelling approaches by providing:

* A **binary response** (`pm10_high`) suitable for **Binomial GLM (logistic regression)**
* A **count response** (`exceed_mtd`) suitable for **Poisson GLM**
* A mix of **continuous and categorical predictors**
* Explicit temporal structure via **cyclic seasonality** (`sin_doy`, `cos_doy`) and weekday effects (`wday`)

Overall, `df_final` meets the professor’s requirements and is appropriate for the intended GLM-based analyses (and extensions).


In [22]:
# Save df_final next to your other outputs in the same project folder
OUTPUT_PATH = "/content/drive/MyDrive/Colab Notebooks/air_quality/df_final.parquet"

df_final.to_parquet(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)

OUTPUT_PATH = "/content/drive/MyDrive/Colab Notebooks/air_quality/df_final.csv"

df_final.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)

Saved to: /content/drive/MyDrive/Colab Notebooks/air_quality/df_final.parquet
Saved to: /content/drive/MyDrive/Colab Notebooks/air_quality/df_final.csv
